# Multimodal Deep Learning for Carotid Artery Atherosclerosis

Notebook này xây dựng một pipeline PyTorch cho bài toán multimodal gồm:

- Nhánh ảnh: mã hóa ảnh carotid để học đặc trưng liên quan đến plaque / echogenicity.
- Nhánh tabular: xử lý dữ liệu lâm sàng và lipid profile.
- Fusion: kết hợp hai nhánh để dự đoán `Plaque_present`.

Lưu ý: dataset mô tả không cung cấp mask phân đoạn, nên notebook này triển khai mô hình phân loại / risk stratification thay vì segmentation pixel-level.

In [29]:
import math
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cuda')

In [30]:
BASE_DIR = Path('/home/jupyter-thinc/Learning/IT2039.CH201')
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)


def find_existing_file(root_dir, filename):
    direct = root_dir / filename
    if direct.exists():
        return direct
    matches = list(root_dir.rglob(filename))
    if matches:
        return matches[0]
    return None


CSV_PATH = find_existing_file(BASE_DIR, 'carotid_clinical_dataset_300cases.csv')
if CSV_PATH is None:
    CSV_PATH = find_existing_file(BASE_DIR, 'carotid_gold_ratio_300cases.csv')

IMAGE_ROOT = None
for candidate in [BASE_DIR / 'datatset' / 'CAROTID_IMAGES', BASE_DIR / 'CAROTID_IMAGES', BASE_DIR / 'carotid_images', BASE_DIR / 'CAROTID_IMAGES'.lower()]:
    if candidate.exists() and candidate.is_dir():
        IMAGE_ROOT = candidate
        break
if IMAGE_ROOT is None:
    for candidate in BASE_DIR.rglob('*'):
        if candidate.is_dir() and candidate.name.lower() == 'carotid_images':
            IMAGE_ROOT = candidate
            break

if CSV_PATH is None:
    raise FileNotFoundError('Khong tim thay file CSV carotid_clinical_dataset_300cases.csv trong workspace.')
if IMAGE_ROOT is None:
    raise FileNotFoundError('Khong tim thay thu muc CAROTID_IMAGES trong workspace.')

print('CSV_PATH =', CSV_PATH)
print('IMAGE_ROOT =', IMAGE_ROOT)

df = pd.read_csv(CSV_PATH)
print(df.shape)
display(df.head())
print(df.columns.tolist())

CSV_PATH = /home/jupyter-thinc/Learning/IT2039.CH201/datatset/carotid_clinical_dataset_300cases.csv
IMAGE_ROOT = /home/jupyter-thinc/Learning/IT2039.CH201/datatset/CAROTID_IMAGES
(300, 15)


,Patient_ID,Age,Sex,Lp(a)_mg_dL,ApoB_mg_dL,LDL_C_mg_dL,Triglyceride_mg_dL,Total_Cholesterol_mg_dL,Non_HDL_mg_dL,IMT_mm,Plaque_present,Plaque_echogenicity,Baseline_Risk_Score,Baseline_Risk_Category,Associated_Images
0,P001,59,Male,10.0,97.7,157.6,123.4,195.3,150.0,0.710,0,NaN,0.49,Low,P001_IMT.png
1,P002,53,Male,37.6,128.0,159.7,152.8,185.8,139.4,0.698,0,NaN,0.11,Low,P002_IMT.png
2,P003,61,Female,18.2,140.4,146.0,86.0,187.1,135.2,0.733,1,Low,0.14,Low,"P003_IMT.png,P003_CCA_L1.png,P003_CCA_L2.png,P..."
3,P004,70,Female,38.2,101.5,101.4,71.1,140.7,102.2,0.879,1,Intermediate,0.00,Low,"P004_IMT.png,P004_CCA_L1.png,P004_CCA_L2.png,P..."
4,P005,52,Female,17.2,99.2,170.8,70.0,186.8,156.9,0.637,0,NaN,0.08,Low,P005_IMT.png


['Patient_ID', 'Age', 'Sex', 'Lp(a)_mg_dL', 'ApoB_mg_dL', 'LDL_C_mg_dL', 'Triglyceride_mg_dL', 'Total_Cholesterol_mg_dL', 'Non_HDL_mg_dL', 'IMT_mm', 'Plaque_present', 'Plaque_echogenicity', 'Baseline_Risk_Score', 'Baseline_Risk_Category', 'Associated_Images']


In [31]:
def find_first_existing_column(frame, candidates):
    normalized = {str(col).strip().lower(): col for col in frame.columns}
    for candidate in candidates:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]
    return None

id_col = find_first_existing_column(df, ['patient_id', 'patientid', 'id', 'case_id', 'caseid', 'subject_id', 'subjectid'])
target_col = find_first_existing_column(df, ['plaque_present', 'target', 'label'])
risk_score_col = find_first_existing_column(df, ['baseline_risk_score', 'risk_score'])
risk_cat_col = find_first_existing_column(df, ['baseline_risk_category', 'risk_category'])
echogenicity_col = find_first_existing_column(df, ['plaque_echogenicity', 'echogenicity'])
image_list_col = find_first_existing_column(df, ['associated_images', 'image_files', 'images'])
sex_col = find_first_existing_column(df, ['sex', 'gender'])
age_col = find_first_existing_column(df, ['age'])
imt_col = find_first_existing_column(df, ['imt'])

if target_col is None:
    raise ValueError('Khong tim thay cot target Plaque_present trong CSV.')

print('id_col =', id_col)
print('target_col =', target_col)
print('risk_score_col =', risk_score_col)
print('risk_cat_col =', risk_cat_col)
print('echogenicity_col =', echogenicity_col)
print('image_list_col =', image_list_col)
print('sex_col =', sex_col)
print('age_col =', age_col)
print('imt_col =', imt_col)

image_index = defaultdict(list)
if IMAGE_ROOT is not None:
    for candidate in IMAGE_ROOT.rglob('*'):
        if candidate.is_file():
            image_index[candidate.name.lower()].append(candidate)
            try:
                relative_key = str(candidate.relative_to(IMAGE_ROOT)).replace('\\', '/').lower()
                image_index[relative_key].append(candidate)
            except ValueError:
                pass

def find_images_for_sample(row):
    if image_list_col is None or image_list_col not in row.index:
        return []
    raw_value = row[image_list_col]
    if pd.isna(raw_value):
        return []

    text = str(raw_value).strip().strip('[]').strip()
    if not text or text.lower() == 'nan':
        return []

    tokens = []
    for chunk in text.replace(';', ',').replace('|', ',').split(','):
        cleaned = chunk.strip().strip("'").strip('"').strip()
        if cleaned:
            tokens.append(cleaned)

    resolved = []
    seen = set()
    for token in tokens:
        candidates = [Path(token)]
        if IMAGE_ROOT is not None:
            candidates.append(IMAGE_ROOT / token)
            candidates.append(IMAGE_ROOT / Path(token).name)
        lookup_keys = [token.lower(), Path(token).name.lower()]
        for key in lookup_keys:
            for candidate in image_index.get(key, []):
                candidates.append(candidate)

        for candidate in candidates:
            if candidate.exists():
                normalized = str(candidate.resolve())
                if normalized not in seen:
                    seen.add(normalized)
                    resolved.append(candidate)
                    break
    return resolved

id_col = Patient_ID
target_col = Plaque_present
risk_score_col = Baseline_Risk_Score
risk_cat_col = Baseline_Risk_Category
echogenicity_col = Plaque_echogenicity
image_list_col = Associated_Images
sex_col = Sex
age_col = Age
imt_col = None


In [32]:
def coerce_sex(series):
    if series is None:
        return None
    mapping = {
        'm': 1.0, 'male': 1.0, '1': 1.0, 1: 1.0,
        'f': 0.0, 'female': 0.0, '0': 0.0, 0: 0.0,
    }
    values = []
    for value in series.fillna('unknown').astype(str).str.strip().str.lower():
        values.append(mapping.get(value, np.nan))
    return pd.Series(values, index=series.index, dtype='float32')


tabular_candidates = [
    'age', 'sex', 'ldl-c', 'ldl_c', 'ldl', 'apob', 'lp(a)', 'lpa', 'triglyceride',
    'total cholesterol', 'total_cholesterol', 'non-hdl', 'non_hdl', 'imt'
]
selected_cols = []
excluded_for_features = {
    str(target_col).strip().lower(),
    str(risk_score_col).strip().lower() if risk_score_col else '',
    str(risk_cat_col).strip().lower() if risk_cat_col else '',
    str(echogenicity_col).strip().lower() if echogenicity_col else '',
    str(image_list_col).strip().lower() if image_list_col else '',
    str(id_col).strip().lower() if id_col else '',
}
for col in df.columns:
    normalized = str(col).strip().lower()
    if normalized in excluded_for_features:
        continue
    if pd.api.types.is_numeric_dtype(df[col]) or normalized in tabular_candidates or normalized.replace(' ', '') in [c.replace(' ', '') for c in tabular_candidates]:
        selected_cols.append(col)

selected_cols = list(dict.fromkeys(selected_cols))
print('Selected tabular columns:', selected_cols)

work_df = df.copy()
if sex_col is not None:
    work_df[sex_col] = coerce_sex(work_df[sex_col])
if risk_score_col is not None:
    work_df[risk_score_col] = pd.to_numeric(work_df[risk_score_col], errors='coerce')
if risk_cat_col is not None:
    work_df[risk_cat_col] = work_df[risk_cat_col].fillna('unknown').astype(str)
if echogenicity_col is not None:
    work_df[echogenicity_col] = work_df[echogenicity_col].fillna('unknown').astype(str)

numeric_cols = []
for col in selected_cols:
    if col == target_col:
        continue
    if col == risk_score_col:
        continue
    if col == risk_cat_col:
        continue
    if col == echogenicity_col:
        continue
    if work_df[col].dtype == 'object':
        continue
    numeric_cols.append(col)

feature_means = work_df[numeric_cols].mean(numeric_only=True)
feature_stds = work_df[numeric_cols].std(numeric_only=True).replace(0, 1.0)
work_df[numeric_cols] = work_df[numeric_cols].fillna(feature_means)
work_df[numeric_cols] = (work_df[numeric_cols] - feature_means) / feature_stds

work_df[target_col] = work_df[target_col].astype(int)
work_df = work_df.reset_index(drop=True)
work_df[[target_col] + ([risk_score_col] if risk_score_col else []) + ([risk_cat_col] if risk_cat_col else []) + ([echogenicity_col] if echogenicity_col else [])].head()

Selected tabular columns: ['Age', 'Sex', 'Lp(a)_mg_dL', 'ApoB_mg_dL', 'LDL_C_mg_dL', 'Triglyceride_mg_dL', 'Total_Cholesterol_mg_dL', 'Non_HDL_mg_dL', 'IMT_mm']


,Plaque_present,Baseline_Risk_Score,Baseline_Risk_Category,Plaque_echogenicity
0,0,0.49,Low,unknown
1,0,0.11,Low,unknown
2,1,0.14,Low,Low
3,1,0.00,Low,Intermediate
4,0,0.08,Low,unknown


In [33]:
class ImageTabularDataset(Dataset):
    def __init__(self, frame, feature_columns, target_col, risk_score_col=None, risk_cat_col=None, echogenicity_col=None, image_size=224, max_images=5):
        self.frame = frame.reset_index(drop=True)
        self.feature_columns = feature_columns
        self.target_col = target_col
        self.risk_score_col = risk_score_col
        self.risk_cat_col = risk_cat_col
        self.echogenicity_col = echogenicity_col
        self.image_size = image_size
        self.max_images = max_images

    def __len__(self):
        return len(self.frame)

    def load_image(self, path):
        image = Image.open(path).convert('RGB').resize((self.image_size, self.image_size))
        array = np.asarray(image, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(array).permute(2, 0, 1)
        tensor = (tensor - 0.5) / 0.5
        return tensor

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        features = torch.tensor(row[self.feature_columns].astype(float).values, dtype=torch.float32)
        label = torch.tensor(float(row[self.target_col]), dtype=torch.float32)

        image_paths = find_images_for_sample(row)
        if len(image_paths) == 0:
            image_tensor = torch.zeros((self.max_images, 3, self.image_size, self.image_size), dtype=torch.float32)
            image_mask = torch.zeros((self.max_images,), dtype=torch.float32)
        else:
            chosen_paths = image_paths[:self.max_images]
            images = [self.load_image(path) for path in chosen_paths]
            image_mask = torch.ones((len(images),), dtype=torch.float32)
            while len(images) < self.max_images:
                images.append(images[-1].clone())
                image_mask = torch.cat([image_mask, torch.zeros((1,), dtype=torch.float32)], dim=0)
            image_tensor = torch.stack(images, dim=0)

        sample = {
            'features': features,
            'images': image_tensor,
            'image_mask': image_mask,
            'label': label,
        }

        if self.risk_score_col is not None and self.risk_score_col in row.index:
            sample['risk_score'] = torch.tensor(float(row[self.risk_score_col]), dtype=torch.float32)
        if self.risk_cat_col is not None and self.risk_cat_col in row.index:
            sample['risk_cat'] = torch.tensor(risk_cat_to_idx[str(row[self.risk_cat_col])], dtype=torch.long)
        if self.echogenicity_col is not None and self.echogenicity_col in row.index:
            sample['echogenicity'] = torch.tensor(echogenicity_to_idx[str(row[self.echogenicity_col])], dtype=torch.long)
        return sample


In [34]:
feature_columns = [col for col in selected_cols if col not in {target_col, risk_score_col, risk_cat_col, echogenicity_col}]
if len(feature_columns) == 0:
    raise ValueError('Khong co feature tabular nao sau khi loc.')

if risk_cat_col is not None:
    risk_categories = sorted(work_df[risk_cat_col].unique().tolist())
    risk_cat_to_idx = {name: idx for idx, name in enumerate(risk_categories)}
else:
    risk_categories = []
    risk_cat_to_idx = {}

echogenicity_categories = []
echogenicity_to_idx = {}
if echogenicity_col is not None:
    echogenicity_categories = sorted(work_df[echogenicity_col].unique().tolist())
    echogenicity_to_idx = {name: idx for idx, name in enumerate(echogenicity_categories)}

train_df, temp_df = train_test_split(
    work_df, test_size=0.30, random_state=SEED, stratify=work_df[target_col]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df[target_col]
)

risk_score_mean = 0.0
risk_score_std = 1.0
if risk_score_col is not None:
    risk_score_mean = float(train_df[risk_score_col].mean())
    risk_score_std = float(train_df[risk_score_col].std()) if float(train_df[risk_score_col].std()) != 0 else 1.0

print('train/val/test =', len(train_df), len(val_df), len(test_df))
print('Train label distribution:', train_df[target_col].value_counts(normalize=True).to_dict())
print('Val label distribution:', val_df[target_col].value_counts(normalize=True).to_dict())
print('Test label distribution:', test_df[target_col].value_counts(normalize=True).to_dict())
print('Risk score mean/std =', risk_score_mean, risk_score_std)
print('Risk categories =', risk_categories)
print('Echogenicity categories =', echogenicity_categories)

train/val/test = 210 45 45
Train label distribution: {0: 0.680952380952381, 1: 0.319047619047619}
Val label distribution: {0: 0.6888888888888889, 1: 0.3111111111111111}
Test label distribution: {0: 0.6888888888888889, 1: 0.3111111111111111}
Risk score mean/std = 0.7228571428571428 0.5887391674123492
Risk categories = ['Low', 'Moderate']
Echogenicity categories = ['High', 'Intermediate', 'Low', 'unknown']


In [35]:
class ImageTabularDataset(Dataset):
    def __init__(self, frame, feature_columns, target_col, risk_score_col=None, risk_score_mean=0.0, risk_score_std=1.0, risk_cat_col=None, echogenicity_col=None, image_size=224, max_images=5):
        self.frame = frame.reset_index(drop=True)
        self.feature_columns = feature_columns
        self.target_col = target_col
        self.risk_score_col = risk_score_col
        self.risk_score_mean = risk_score_mean
        self.risk_score_std = risk_score_std if risk_score_std not in {0, None} else 1.0
        self.risk_cat_col = risk_cat_col
        self.echogenicity_col = echogenicity_col
        self.image_size = image_size
        self.max_images = max_images

    def __len__(self):
        return len(self.frame)

    def load_image(self, path):
        image = Image.open(path).convert('RGB').resize((self.image_size, self.image_size))
        array = np.asarray(image, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(array).permute(2, 0, 1)
        tensor = (tensor - 0.5) / 0.5
        return tensor

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        features = torch.tensor(row[self.feature_columns].astype(float).values, dtype=torch.float32)
        label = torch.tensor(float(row[self.target_col]), dtype=torch.float32)

        image_paths = find_images_for_sample(row)
        if len(image_paths) == 0:
            image_tensor = torch.zeros((self.max_images, 3, self.image_size, self.image_size), dtype=torch.float32)
            image_mask = torch.zeros((self.max_images,), dtype=torch.float32)
        else:
            chosen_paths = image_paths[:self.max_images]
            images = [self.load_image(path) for path in chosen_paths]
            image_mask = torch.ones((len(images),), dtype=torch.float32)
            while len(images) < self.max_images:
                images.append(images[-1].clone())
                image_mask = torch.cat([image_mask, torch.zeros((1,), dtype=torch.float32)], dim=0)
            image_tensor = torch.stack(images, dim=0)

        sample = {
            'features': features,
            'images': image_tensor,
            'image_mask': image_mask,
            'label': label,
        }

        if self.risk_score_col is not None and self.risk_score_col in row.index:
            raw_risk_score = float(row[self.risk_score_col])
            normalized_risk_score = (raw_risk_score - self.risk_score_mean) / self.risk_score_std
            sample['risk_score'] = torch.tensor(normalized_risk_score, dtype=torch.float32)
        if self.risk_cat_col is not None and self.risk_cat_col in row.index:
            sample['risk_cat'] = torch.tensor(risk_cat_to_idx[str(row[self.risk_cat_col])], dtype=torch.long)
        if self.echogenicity_col is not None and self.echogenicity_col in row.index:
            sample['echogenicity'] = torch.tensor(echogenicity_to_idx[str(row[self.echogenicity_col])], dtype=torch.long)
        return sample

train_dataset = ImageTabularDataset(train_df, feature_columns, target_col, risk_score_col=risk_score_col, risk_score_mean=risk_score_mean, risk_score_std=risk_score_std, risk_cat_col=risk_cat_col, echogenicity_col=echogenicity_col)
val_dataset = ImageTabularDataset(val_df, feature_columns, target_col, risk_score_col=risk_score_col, risk_score_mean=risk_score_mean, risk_score_std=risk_score_std, risk_cat_col=risk_cat_col, echogenicity_col=echogenicity_col)
test_dataset = ImageTabularDataset(test_df, feature_columns, target_col, risk_score_col=risk_score_col, risk_score_mean=risk_score_mean, risk_score_std=risk_score_std, risk_cat_col=risk_cat_col, echogenicity_col=echogenicity_col)

train_dataset[0].keys()

dict_keys(['features', 'images', 'image_mask', 'label', 'risk_score', 'risk_cat', 'echogenicity'])

In [36]:
class SimpleCNNEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )

    def forward(self, images):
        return self.proj(self.features(images))


class TabularEncoder(nn.Module):
    def __init__(self, input_dim, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(out_dim, out_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

In [37]:
class SimpleCNNEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )

    def forward(self, images):
        return self.proj(self.features(images))


class TabularEncoder(nn.Module):
    def __init__(self, input_dim, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(out_dim, out_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class MultimodalTaskModel(nn.Module):
    def __init__(self, tabular_dim, task_type, image_embed_dim=128, tabular_embed_dim=128, fusion_dim=128, num_classes=None):
        super().__init__()
        self.task_type = task_type
        self.image_encoder = SimpleCNNEncoder(out_dim=image_embed_dim)
        self.tabular_encoder = TabularEncoder(tabular_dim, out_dim=tabular_embed_dim)
        self.shared_dim = image_embed_dim + tabular_embed_dim
        self.fusion = nn.Sequential(
            nn.Linear(self.shared_dim, fusion_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
        )

        if task_type in {'binary', 'regression'}:
            self.head = nn.Linear(fusion_dim, 1)
        elif task_type == 'multiclass':
            if num_classes is None or num_classes < 2:
                raise ValueError('multiclass task requires num_classes >= 2')
            self.head = nn.Linear(fusion_dim, num_classes)
        else:
            raise ValueError(f'Unsupported task_type: {task_type}')

    def forward(self, features, images, image_mask=None):
        batch_size, num_images, channels, height, width = images.shape
        flat_images = images.view(batch_size * num_images, channels, height, width)
        image_embeddings = self.image_encoder(flat_images).view(batch_size, num_images, -1)

        if image_mask is not None:
            weights = image_mask.unsqueeze(-1)
            summed = (image_embeddings * weights).sum(dim=1)
            denom = weights.sum(dim=1).clamp_min(1.0)
            image_embedding = summed / denom
        else:
            image_embedding = image_embeddings.mean(dim=1)

        tabular_embedding = self.tabular_encoder(features)
        fused = torch.cat([image_embedding, tabular_embedding], dim=-1)
        fused = self.fusion(fused)
        output = self.head(fused)

        if self.task_type in {'binary', 'regression'}:
            return output.squeeze(-1)
        return output


plaque_model = MultimodalTaskModel(
    tabular_dim=len(feature_columns),
    task_type='binary',
).to(DEVICE)

risk_score_model = MultimodalTaskModel(
    tabular_dim=len(feature_columns),
    task_type='regression',
).to(DEVICE)

risk_category_model = None
if len(risk_categories) > 0:
    risk_category_model = MultimodalTaskModel(
        tabular_dim=len(feature_columns),
        task_type='multiclass',
        num_classes=len(risk_categories),
    ).to(DEVICE)

echogenicity_model = None
if len(echogenicity_categories) > 0:
    echogenicity_model = MultimodalTaskModel(
        tabular_dim=len(feature_columns),
        task_type='multiclass',
        num_classes=len(echogenicity_categories),
    ).to(DEVICE)

task_models = {
    'plaque': plaque_model,
    'risk_score': risk_score_model,
    'risk_category': risk_category_model,
    'echogenicity': echogenicity_model,
}

model = plaque_model
model

MultimodalTaskModel(
  (image_encoder): SimpleCNNEncoder(
    (features): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (6): ReLU(inplace=True)
      (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (10): ReLU(inplace=True)
      (11): AdaptiveAvgPool2d(output_size=(1, 1))
    )
    (proj): Sequential(
      (0): Flatten(start_dim=1, e

In [38]:
def collate_batch(batch):
    features = torch.stack([item['features'] for item in batch], dim=0)
    images = torch.stack([item['images'] for item in batch], dim=0)
    image_mask = torch.stack([item['image_mask'] for item in batch], dim=0)
    labels = torch.stack([item['label'] for item in batch], dim=0)
    result = {
        'features': features,
        'images': images,
        'image_mask': image_mask,
        'label': labels,
    }
    if 'risk_score' in batch[0]:
        result['risk_score'] = torch.stack([item['risk_score'] for item in batch], dim=0)
    if 'risk_cat' in batch[0]:
        result['risk_cat'] = torch.stack([item['risk_cat'] for item in batch], dim=0)
    if 'echogenicity' in batch[0]:
        result['echogenicity'] = torch.stack([item['echogenicity'] for item in batch], dim=0)
    return result


def make_weighted_sampler(frame):
    class_counts = frame[target_col].value_counts().to_dict()
    sample_weights = frame[target_col].map(lambda label: 1.0 / class_counts[label]).values
    sample_weights = torch.as_tensor(sample_weights, dtype=torch.double)
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=8, sampler=make_weighted_sampler(train_df),
    num_workers=0, collate_fn=collate_batch
)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0, collate_fn=collate_batch)

batch = next(iter(train_loader))
{k: tuple(v.shape) for k, v in batch.items()}

{'features': (8, 9),
 'images': (8, 5, 3, 224, 224),
 'image_mask': (8, 5),
 'label': (8,),
 'risk_score': (8,),
 'risk_cat': (8,),
 'echogenicity': (8,)}

In [39]:
plaque_criterion = nn.BCEWithLogitsLoss()
risk_score_criterion = nn.SmoothL1Loss()
aux_risk_criterion = nn.CrossEntropyLoss() if len(risk_categories) > 0 else None
aux_echogenicity_criterion = nn.CrossEntropyLoss() if len(echogenicity_categories) > 0 else None


def sigmoid_probs(logits):
    return torch.sigmoid(logits).detach().cpu().numpy()


def denormalize_risk_score(score_tensor):
    return score_tensor * risk_score_std + risk_score_mean


def make_task_loaders(task_name, batch_size=8):
    if task_name == 'plaque':
        train_sampler = make_weighted_sampler(train_df)
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=train_sampler,
            num_workers=0,
            collate_fn=collate_batch,
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            collate_fn=collate_batch,
        )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_batch,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_batch,
    )
    return train_loader, val_loader, test_loader


def get_task_outputs(model, batch, task_name):
    features = batch['features'].to(DEVICE)
    images = batch['images'].to(DEVICE)
    image_mask = batch['image_mask'].to(DEVICE)
    outputs = model(features, images, image_mask)
    if task_name == 'plaque':
        return outputs, batch['label'].to(DEVICE)
    if task_name == 'risk_score':
        return outputs, batch['risk_score'].to(DEVICE)
    if task_name == 'risk_category':
        return outputs, batch['risk_cat'].to(DEVICE)
    if task_name == 'echogenicity':
        return outputs, batch['echogenicity'].to(DEVICE)
    raise ValueError(f'Unsupported task_name: {task_name}')


def evaluate_task(model, loader, task_name):
    model.eval()
    total_loss = 0.0
    total_count = 0

    if task_name == 'plaque':
        all_targets = []
        all_probs = []
        all_preds = []
        with torch.no_grad():
            for batch in loader:
                logits, targets = get_task_outputs(model, batch, task_name)
                loss = plaque_criterion(logits, targets)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).long()
                bs = targets.size(0)
                total_loss += loss.item() * bs
                total_count += bs
                all_targets.extend(targets.long().cpu().tolist())
                all_probs.extend(probs.cpu().tolist())
                all_preds.extend(preds.cpu().tolist())

        avg_loss = total_loss / max(total_count, 1)
        acc = accuracy_score(all_targets, all_preds)
        f1 = f1_score(all_targets, all_preds, zero_division=0)
        try:
            auc = roc_auc_score(all_targets, all_probs)
        except ValueError:
            auc = float('nan')
        return {
            'loss': avg_loss,
            'acc': acc,
            'f1': f1,
            'auc': auc,
            'primary_score': auc if not math.isnan(auc) else acc,
            'targets': all_targets,
            'preds': all_preds,
            'probs': all_probs,
        }

    if task_name == 'risk_score':
        all_true_raw = []
        all_pred_raw = []
        with torch.no_grad():
            for batch in loader:
                preds, targets = get_task_outputs(model, batch, task_name)
                loss = risk_score_criterion(preds, targets)
                pred_raw = denormalize_risk_score(preds)
                target_raw = denormalize_risk_score(targets)
                bs = targets.size(0)
                total_loss += loss.item() * bs
                total_count += bs
                all_true_raw.extend(target_raw.cpu().tolist())
                all_pred_raw.extend(pred_raw.cpu().tolist())

        avg_loss = total_loss / max(total_count, 1)
        mae = float(np.mean(np.abs(np.asarray(all_pred_raw) - np.asarray(all_true_raw)))) if all_true_raw else float('nan')
        rmse = float(np.sqrt(np.mean((np.asarray(all_pred_raw) - np.asarray(all_true_raw)) ** 2))) if all_true_raw else float('nan')
        return {
            'loss': avg_loss,
            'mae': mae,
            'rmse': rmse,
            'primary_score': -mae if not math.isnan(mae) else -avg_loss,
            'targets': all_true_raw,
            'preds': all_pred_raw,
        }

    all_targets = []
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            logits, targets = get_task_outputs(model, batch, task_name)
            loss = aux_risk_criterion(logits, targets) if task_name == 'risk_category' else aux_echogenicity_criterion(logits, targets)
            preds = torch.argmax(logits, dim=-1)
            bs = targets.size(0)
            total_loss += loss.item() * bs
            total_count += bs
            all_targets.extend(targets.long().cpu().tolist())
            all_preds.extend(preds.long().cpu().tolist())

    avg_loss = total_loss / max(total_count, 1)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return {
        'loss': avg_loss,
        'acc': acc,
        'f1_macro': f1,
        'primary_score': acc,
        'targets': all_targets,
        'preds': all_preds,
    }


def train_task(task_name, model, train_loader, val_loader, epochs=15):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)
    best_score = -float('inf')
    best_path = OUTPUT_DIR / f'best_{task_name}_model.pt'
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        count = 0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            outputs, targets = get_task_outputs(model, batch, task_name)

            if task_name == 'plaque':
                loss = plaque_criterion(outputs, targets)
            elif task_name == 'risk_score':
                loss = risk_score_criterion(outputs, targets)
            elif task_name == 'risk_category':
                loss = aux_risk_criterion(outputs, targets)
            elif task_name == 'echogenicity':
                loss = aux_echogenicity_criterion(outputs, targets)
            else:
                raise ValueError(f'Unsupported task_name: {task_name}')

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            bs = targets.size(0)
            running_loss += loss.item() * bs
            count += bs

        train_loss = running_loss / max(count, 1)
        val_metrics = evaluate_task(model, val_loader, task_name)
        scheduler.step(val_metrics['primary_score'])

        if val_metrics['primary_score'] > best_score:
            best_score = val_metrics['primary_score']
            checkpoint = {
                'model_state_dict': model.state_dict(),
                'task_name': task_name,
                'feature_columns': feature_columns,
                'target_col': target_col,
                'risk_score_mean': risk_score_mean,
                'risk_score_std': risk_score_std,
                'feature_means': feature_means.to_dict(),
                'feature_stds': feature_stds.to_dict(),
                'risk_categories': risk_categories,
                'echogenicity_categories': echogenicity_categories,
            }
            torch.save(checkpoint, best_path)

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_metrics['loss'],
            'val_primary_score': val_metrics['primary_score'],
        }
        if task_name == 'plaque':
            row.update({'val_acc': val_metrics['acc'], 'val_f1': val_metrics['f1'], 'val_auc': val_metrics['auc']})
        elif task_name == 'risk_score':
            row.update({'val_mae': val_metrics['mae'], 'val_rmse': val_metrics['rmse']})
        else:
            row.update({'val_acc': val_metrics['acc'], 'val_f1_macro': val_metrics['f1_macro']})
        history.append(row)
        print(f'[{task_name}]', row)

    history_df = pd.DataFrame(history)
    return best_path, history_df


def run_all_task_training(epochs=15):
    task_configs = {
        'plaque': {
            'enabled': True,
            'model': plaque_model,
        },
        'risk_score': {
            'enabled': True,
            'model': risk_score_model,
        },
        'risk_category': {
            'enabled': risk_category_model is not None,
            'model': risk_category_model,
        },
        'echogenicity': {
            'enabled': echogenicity_model is not None,
            'model': echogenicity_model,
        },
    }

    task_runs = {}
    for task_name, task_cfg in task_configs.items():
        if not task_cfg['enabled'] or task_cfg['model'] is None:
            print(f'Skip {task_name}: missing labels or model.')
            continue

        train_loader_task, val_loader_task, test_loader_task = make_task_loaders(task_name)
        best_path, history_df = train_task(task_name, task_cfg['model'], train_loader_task, val_loader_task, epochs=epochs)
        task_runs[task_name] = {
            'best_path': best_path,
            'history': history_df,
            'test_loader': test_loader_task,
        }
        display(history_df)
        print('Best checkpoint:', best_path)

    return task_runs


task_configs = {
    'plaque': {
        'enabled': True,
        'model': plaque_model,
    },
    'risk_score': {
        'enabled': True,
        'model': risk_score_model,
    },
    'risk_category': {
        'enabled': risk_category_model is not None,
        'model': risk_category_model,
    },
    'echogenicity': {
        'enabled': echogenicity_model is not None,
        'model': echogenicity_model,
    },
}

task_configs

{'plaque': {'enabled': True,
  'model': MultimodalTaskModel(
    (image_encoder): SimpleCNNEncoder(
      (features): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (6): ReLU(inplace=True)
        (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (10): ReLU(inplace=True)
        (11): AdaptiveAvgPool2d(output_size=(1,

In [40]:
def evaluate_saved_task_models():
    results = {}
    for task_name, task_cfg in task_configs.items():
        if not task_cfg['enabled'] or task_cfg['model'] is None:
            print(f'Skip {task_name}: missing labels or model.')
            continue

        best_path = OUTPUT_DIR / f'best_{task_name}_model.pt'
        if not best_path.exists():
            print(f'Skip {task_name}: checkpoint not found at {best_path}')
            continue

        ckpt = torch.load(best_path, map_location=DEVICE)
        task_cfg['model'].load_state_dict(ckpt['model_state_dict'])
        _, _, test_loader_task = make_task_loaders(task_name)
        metrics = evaluate_task(task_cfg['model'], test_loader_task, task_name)
        results[task_name] = metrics

        summary = {k: v for k, v in metrics.items() if k not in {'targets', 'preds', 'probs'}}
        print(f'[{task_name}]', summary)
        if task_name == 'plaque':
            print('Confusion matrix:')
            print(confusion_matrix(metrics['targets'], metrics['preds']))
            print('Classification report:')
            print(classification_report(metrics['targets'], metrics['preds'], zero_division=0))
        elif task_name == 'risk_score':
            print('Risk score MAE:', metrics['mae'])
            print('Risk score RMSE:', metrics['rmse'])
        else:
            print('Classification report:')
            print(classification_report(metrics['targets'], metrics['preds'], zero_division=0))

    return results


test_metrics_by_task = evaluate_saved_task_models()
test_metrics_by_task

Skip plaque: checkpoint not found at /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_plaque_model.pt
Skip risk_score: checkpoint not found at /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_risk_score_model.pt
Skip risk_category: checkpoint not found at /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_risk_category_model.pt
Skip echogenicity: checkpoint not found at /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_echogenicity_model.pt


{}

In [41]:
def evaluate_all_models_on_test():
    summary_rows = []
    detailed_results = {}

    for task_name in ['plaque', 'risk_score', 'risk_category', 'echogenicity']:
        task_cfg = task_configs.get(task_name)
        if task_cfg is None or not task_cfg['enabled'] or task_cfg['model'] is None:
            print(f'Skip {task_name}: missing labels or model.')
            continue

        checkpoint_path = OUTPUT_DIR / f'best_{task_name}_model.pt'
        if not checkpoint_path.exists():
            print(f'Skip {task_name}: checkpoint not found at {checkpoint_path}')
            continue

        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        task_cfg['model'].load_state_dict(ckpt['model_state_dict'])
        _, _, test_loader_task = make_task_loaders(task_name)
        metrics = evaluate_task(task_cfg['model'], test_loader_task, task_name)
        detailed_results[task_name] = metrics

        summary = {'task_name': task_name, 'checkpoint': str(checkpoint_path)}
        if task_name == 'plaque':
            summary.update({
                'loss': metrics['loss'],
                'acc': metrics['acc'],
                'f1': metrics['f1'],
                'auc': metrics['auc'],
            })
        elif task_name == 'risk_score':
            summary.update({
                'loss': metrics['loss'],
                'mae': metrics['mae'],
                'rmse': metrics['rmse'],
            })
        else:
            summary.update({
                'loss': metrics['loss'],
                'acc': metrics['acc'],
                'f1_macro': metrics['f1_macro'],
            })
        summary_rows.append(summary)

        print(f'[{task_name}]')
        if task_name == 'plaque':
            print('Confusion matrix:')
            print(confusion_matrix(metrics['targets'], metrics['preds']))
            print('Classification report:')
            print(classification_report(metrics['targets'], metrics['preds'], zero_division=0))
        elif task_name == 'risk_score':
            print('Risk score MAE:', metrics['mae'])
            print('Risk score RMSE:', metrics['rmse'])
        else:
            print('Classification report:')
            print(classification_report(metrics['targets'], metrics['preds'], zero_division=0))

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        display(summary_df)
    return detailed_results, summary_df


test_results_by_task, test_results_df = evaluate_all_models_on_test()
test_results_df

Skip plaque: checkpoint not found at /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_plaque_model.pt
Skip risk_score: checkpoint not found at /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_risk_score_model.pt
Skip risk_category: checkpoint not found at /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_risk_category_model.pt
Skip echogenicity: checkpoint not found at /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_echogenicity_model.pt


""


In [42]:
def predict_one(task_name='plaque', sample_index=0, dataset=test_dataset):
    if task_name not in task_configs:
        raise ValueError(f'Unknown task_name: {task_name}')

    task_cfg = task_configs[task_name]
    if not task_cfg['enabled'] or task_cfg['model'] is None:
        raise ValueError(f'Task {task_name} is not enabled in this notebook.')

    checkpoint_path = OUTPUT_DIR / f'best_{task_name}_model.pt'
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}. Run run_all_task_training() first.')

    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    task_cfg['model'].load_state_dict(ckpt['model_state_dict'])
    task_cfg['model'].eval()

    sample = dataset[sample_index]
    with torch.no_grad():
        outputs = task_cfg['model'](
            sample['features'].unsqueeze(0).to(DEVICE),
            sample['images'].unsqueeze(0).to(DEVICE),
            sample['image_mask'].unsqueeze(0).to(DEVICE),
        )

        result = {
            'task_name': task_name,
            'sample_index': sample_index,
        }

        if task_name == 'plaque':
            plaque_prob = torch.sigmoid(outputs).item()
            result.update({
                'pred_prob_plaque': plaque_prob,
                'pred_label_plaque': int(plaque_prob >= 0.5),
                'true_label_plaque': int(sample['label'].item()),
            })
        elif task_name == 'risk_score':
            predicted_risk_score = (outputs.item() * risk_score_std) + risk_score_mean
            result.update({
                'pred_baseline_risk_score': predicted_risk_score,
                'true_baseline_risk_score': (sample['risk_score'].item() * risk_score_std) + risk_score_mean if 'risk_score' in sample else None,
            })
        elif task_name == 'risk_category':
            risk_category_idx = int(torch.argmax(outputs, dim=-1).item())
            result['pred_baseline_risk_category'] = risk_categories[risk_category_idx]
            if 'risk_cat' in sample:
                result['true_baseline_risk_category'] = risk_categories[int(sample['risk_cat'].item())]
        elif task_name == 'echogenicity':
            echogenicity_idx = int(torch.argmax(outputs, dim=-1).item())
            result['pred_plaque_echogenicity'] = echogenicity_categories[echogenicity_idx]
            if 'echogenicity' in sample:
                result['true_plaque_echogenicity'] = echogenicity_categories[int(sample['echogenicity'].item())]

    return result


predict_one

<function __main__.predict_one(task_name='plaque', sample_index=0, dataset=<__main__.ImageTabularDataset object at 0x70d4ec343ef0>)>

In [43]:
task_runs = run_all_task_training(epochs=15)
task_runs

[plaque] {'epoch': 1, 'train_loss': 0.5338641195070176, 'val_loss': 1.3972990459865995, 'val_primary_score': 0.8018433179723502, 'val_acc': 0.3111111111111111, 'val_f1': 0.4745762711864407, 'val_auc': 0.8018433179723502}
[plaque] {'epoch': 2, 'train_loss': 0.10502719288425787, 'val_loss': 0.0748831702189313, 'val_primary_score': 1.0, 'val_acc': 0.9777777777777777, 'val_f1': 0.9629629629629629, 'val_auc': 1.0}
[plaque] {'epoch': 3, 'train_loss': 0.004239178285934031, 'val_loss': 0.001700972496635384, 'val_primary_score': 1.0, 'val_acc': 1.0, 'val_f1': 1.0, 'val_auc': 1.0}
[plaque] {'epoch': 4, 'train_loss': 0.016551831386806, 'val_loss': 0.0006048467993322345, 'val_primary_score': 1.0, 'val_acc': 1.0, 'val_f1': 1.0, 'val_auc': 1.0}
[plaque] {'epoch': 5, 'train_loss': 0.05286353136484866, 'val_loss': 0.0007439666117231051, 'val_primary_score': 1.0, 'val_acc': 1.0, 'val_f1': 1.0, 'val_auc': 1.0}
[plaque] {'epoch': 6, 'train_loss': 0.000863148111819187, 'val_loss': 0.00031151718964489796, 

,epoch,train_loss,val_loss,val_primary_score,val_acc,val_f1,val_auc
0,1,0.533864,1.397299,0.801843,0.311111,0.474576,0.801843
1,2,0.105027,0.074883,1.000000,0.977778,0.962963,1.000000
2,3,0.004239,0.001701,1.000000,1.000000,1.000000,1.000000
3,4,0.016552,0.000605,1.000000,1.000000,1.000000,1.000000
4,5,0.052864,0.000744,1.000000,1.000000,1.000000,1.000000
5,6,0.000863,0.000312,1.000000,1.000000,1.000000,1.000000
6,7,0.000458,0.000117,1.000000,1.000000,1.000000,1.000000
7,8,0.000326,0.000161,1.000000,1.000000,1.000000,1.000000
8,9,0.000238,0.000128,1.000000,1.000000,1.000000,1.000000
9,10,0.000887,0.000155,1.000000,1.000000,1.000000,1.000000


Best checkpoint: /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_plaque_model.pt
[risk_score] {'epoch': 1, 'train_loss': 0.4229675128346398, 'val_loss': 0.5163599610328674, 'val_primary_score': -0.5599353704187605, 'val_mae': 0.5599353704187605, 'val_rmse': 0.6360541453583588}
[risk_score] {'epoch': 2, 'train_loss': 0.343518420628139, 'val_loss': 0.4096486634678311, 'val_primary_score': -0.4613556550608741, 'val_mae': 0.4613556550608741, 'val_rmse': 0.5666129630363351}
[risk_score] {'epoch': 3, 'train_loss': 0.3264170970235552, 'val_loss': 0.41758817036946616, 'val_primary_score': -0.45705476933055456, 'val_mae': 0.45705476933055456, 'val_rmse': 0.5768678370517092}
[risk_score] {'epoch': 4, 'train_loss': 0.29977462064652216, 'val_loss': 0.42812305291493735, 'val_primary_score': -0.46955743034680686, 'val_mae': 0.46955743034680686, 'val_rmse': 0.58229254249425}
[risk_score] {'epoch': 5, 'train_loss': 0.28888934957129614, 'val_loss': 0.46291038195292156, 'val_primary_score': -0.49

,epoch,train_loss,val_loss,val_primary_score,val_mae,val_rmse
0,1,0.422968,0.516360,-0.559935,0.559935,0.636054
1,2,0.343518,0.409649,-0.461356,0.461356,0.566613
2,3,0.326417,0.417588,-0.457055,0.457055,0.576868
3,4,0.299775,0.428123,-0.469557,0.469557,0.582293
4,5,0.288889,0.462910,-0.493518,0.493518,0.607795
5,6,0.270203,0.452661,-0.482988,0.482988,0.602580
6,7,0.267786,0.440209,-0.477111,0.477111,0.594189
7,8,0.262885,0.470974,-0.491357,0.491357,0.617804
8,9,0.259533,0.455376,-0.482354,0.482354,0.610699
9,10,0.245615,0.467881,-0.490866,0.490866,0.617042


Best checkpoint: /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_risk_score_model.pt
[risk_category] {'epoch': 1, 'train_loss': 0.26928891079677714, 'val_loss': 0.111988672034608, 'val_primary_score': 0.9777777777777777, 'val_acc': 0.9777777777777777, 'val_f1_macro': 0.4943820224719101}
[risk_category] {'epoch': 2, 'train_loss': 0.1496153270204862, 'val_loss': 0.11454860470775101, 'val_primary_score': 0.9777777777777777, 'val_acc': 0.9777777777777777, 'val_f1_macro': 0.4943820224719101}
[risk_category] {'epoch': 3, 'train_loss': 0.09847121458678018, 'val_loss': 0.11508999951183796, 'val_primary_score': 0.9777777777777777, 'val_acc': 0.9777777777777777, 'val_f1_macro': 0.4943820224719101}
[risk_category] {'epoch': 4, 'train_loss': 0.11586412902744044, 'val_loss': 0.1399061817054947, 'val_primary_score': 0.9777777777777777, 'val_acc': 0.9777777777777777, 'val_f1_macro': 0.4943820224719101}
[risk_category] {'epoch': 5, 'train_loss': 0.08919934589593183, 'val_loss': 0.12484330079621

,epoch,train_loss,val_loss,val_primary_score,val_acc,val_f1_macro
0,1,0.269289,0.111989,0.977778,0.977778,0.494382
1,2,0.149615,0.114549,0.977778,0.977778,0.494382
2,3,0.098471,0.115090,0.977778,0.977778,0.494382
3,4,0.115864,0.139906,0.977778,0.977778,0.494382
4,5,0.089199,0.124843,0.977778,0.977778,0.494382
5,6,0.080224,0.135157,0.977778,0.977778,0.494382
6,7,0.081225,0.139286,0.977778,0.977778,0.494382
7,8,0.080900,0.137261,0.977778,0.977778,0.494382
8,9,0.064352,0.147574,0.977778,0.977778,0.494382
9,10,0.070074,0.146968,0.977778,0.977778,0.494382


Best checkpoint: /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_risk_category_model.pt
[echogenicity] {'epoch': 1, 'train_loss': 0.9312520296091125, 'val_loss': 1.0320880479282803, 'val_primary_score': 0.6888888888888889, 'val_acc': 0.6888888888888889, 'val_f1_macro': 0.20394736842105263}
[echogenicity] {'epoch': 2, 'train_loss': 0.5052087693696931, 'val_loss': 0.6026389480464989, 'val_primary_score': 0.8, 'val_acc': 0.8, 'val_f1_macro': 0.42024886877828055}
[echogenicity] {'epoch': 3, 'train_loss': 0.3122034839221409, 'val_loss': 0.8408657590548198, 'val_primary_score': 0.7555555555555555, 'val_acc': 0.7555555555555555, 'val_f1_macro': 0.4903343782654127}
[echogenicity] {'epoch': 4, 'train_loss': 0.14471711466709772, 'val_loss': 0.029736298167457185, 'val_primary_score': 1.0, 'val_acc': 1.0, 'val_f1_macro': 1.0}
[echogenicity] {'epoch': 5, 'train_loss': 0.09465759807665433, 'val_loss': 0.006332165561616421, 'val_primary_score': 1.0, 'val_acc': 1.0, 'val_f1_macro': 1.0}
[echoge

,epoch,train_loss,val_loss,val_primary_score,val_acc,val_f1_macro
0,1,0.931252,1.032088,0.688889,0.688889,0.203947
1,2,0.505209,0.602639,0.800000,0.800000,0.420249
2,3,0.312203,0.840866,0.755556,0.755556,0.490334
3,4,0.144717,0.029736,1.000000,1.000000,1.000000
4,5,0.094658,0.006332,1.000000,1.000000,1.000000
5,6,0.015716,0.007112,1.000000,1.000000,1.000000
6,7,0.007701,0.000996,1.000000,1.000000,1.000000
7,8,0.010628,0.000960,1.000000,1.000000,1.000000
8,9,0.006089,0.001059,1.000000,1.000000,1.000000
9,10,0.003817,0.000876,1.000000,1.000000,1.000000


Best checkpoint: /home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_echogenicity_model.pt


{'plaque': {'best_path': PosixPath('/home/jupyter-thinc/Learning/IT2039.CH201/outputs/best_plaque_model.pt'),
  'history':     epoch  train_loss  val_loss  val_primary_score   val_acc    val_f1  \
  0       1    0.533864  1.397299           0.801843  0.311111  0.474576   
  1       2    0.105027  0.074883           1.000000  0.977778  0.962963   
  2       3    0.004239  0.001701           1.000000  1.000000  1.000000   
  3       4    0.016552  0.000605           1.000000  1.000000  1.000000   
  4       5    0.052864  0.000744           1.000000  1.000000  1.000000   
  5       6    0.000863  0.000312           1.000000  1.000000  1.000000   
  6       7    0.000458  0.000117           1.000000  1.000000  1.000000   
  7       8    0.000326  0.000161           1.000000  1.000000  1.000000   
  8       9    0.000238  0.000128           1.000000  1.000000  1.000000   
  9      10    0.000887  0.000155           1.000000  1.000000  1.000000   
  10     11    0.030230  0.000140          